# TV3 — Random Forest (W3-03 / W3-04)

Role: Random Forest Lead. Data đã được nhóm chia + tiền xử lý xong. Notebook này **không** chia lại dữ liệu.

| Đầu việc | Việc làm trong notebook |
|---|---|
| W2-02 | Đã xong trên repo (`data/processed/random_split_v1/` + `data/model_ready/random/`). Không chạy lại. |
| W3-03 | Train/tune RF trên Time-based split (`data/model_ready/time/with_port/`) |
| W3-04 | Train/tune RF trên Random split đối chứng (`data/model_ready/random/with_port/`) |

Quy tắc:
- Chỉ **fit Train, chọn tham số trên Validation**.
- **Không đọc Test** (`X_test.csv` / `y_test.csv`) — Test khóa đến W3-09 (TV5).
- `class_weight='balanced'`, seed 42. Không SMOTE (TV4).
- Kết quả chính là Time-based; Random chỉ đối chứng.

Giới hạn đã ghi trong `split_metadata.json`: Train chỉ có FTP-Patator (không có SSH); Test chỉ có SSH (không có FTP). Validation có cả hai. Đây là thiết kế zero-shot, không phải lỗi chia data.

In [1]:
from pathlib import Path
import json, time, itertools
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, confusion_matrix,
)

# Notebook đặt trong notebooks/ — root repo là thư mục cha
ROOT = Path.cwd()
if not (ROOT / "data" / "model_ready").exists():
    ROOT = Path.cwd().parent
assert (ROOT / "data" / "model_ready" / "time" / "with_port" / "X_train.csv").exists(), (
    f"Không thấy data/model_ready. cwd={Path.cwd()} ROOT={ROOT}"
)

SEED = 42
OUT = ROOT / "artifacts" / "TV3_random_forest"
OUT.mkdir(parents=True, exist_ok=True)

# Kịch bản bắt buộc W3-03 / W3-04. with_port = kết quả chính.
SCENARIOS = [
    {"split": "time",   "scenario": "with_port", "job": "W3-03"},
    {"split": "random", "scenario": "with_port", "job": "W3-04"},
]

# Lưới nhỏ quanh preset nhóm trong configs/experiment.yaml
# (n_estimators=150, max_depth=16, min_samples_leaf=5, max_features=sqrt)
GRID = {
    "n_estimators": [100, 150, 300],
    "max_depth": [10, 16, None],
    "min_samples_leaf": [5],
}

print("ROOT =", ROOT)
print("OUT  =", OUT)
print("Số cấu hình mỗi split:", np.prod([len(v) for v in GRID.values()]))

ROOT = c:\Users\Dung\cicids-bruteforce-ids
OUT  = c:\Users\Dung\cicids-bruteforce-ids\artifacts\TV3_random_forest
Số cấu hình mỗi split: 9


## Load Train + Validation (không load Test)

`y_*.csv` có cột `BinaryLabel` (0 = Normal, 1 = Attack). `y_*_subtype.csv` có thêm `Subtype` để tính recall FTP/SSH trên Validation.

In [2]:
def load_xy(split: str, scenario: str, part: str):
    """part: train | validation. Cấm 'test'."""
    if part == "test":
        raise RuntimeError("Không được load Test ở giai đoạn tune (W3-09 / TV5).")
    base = ROOT / "data" / "model_ready" / split / scenario
    X = pd.read_csv(base / f"X_{part}.csv")
    y = pd.read_csv(base / f"y_{part}.csv")["BinaryLabel"].astype(int)
    sub = pd.read_csv(base / f"y_{part}_subtype.csv")
    assert len(X) == len(y) == len(sub)
    assert list(X.columns)  # nonempty
    if X.isna().any().any() or np.isinf(X.to_numpy(dtype=float)).any():
        raise RuntimeError(f"NaN/Inf trong {split}/{scenario}/{part}")
    return X, y, sub["Subtype"]


def attack_metrics(y_true, y_pred, scores=None):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_attack": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall_attack": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_attack": float(f1_score(y_true, y_pred, zero_division=0)),
        "fpr": float(fp / (fp + tn)) if (fp + tn) else None,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "n": int(len(y_true)),
        "n_attack": int(y_true.sum()),
    }
    if scores is not None and len(np.unique(y_true)) == 2:
        out["average_precision"] = float(average_precision_score(y_true, scores))
        out["roc_auc"] = float(roc_auc_score(y_true, scores))
    return out


def subtype_recall(subtypes, y_pred):
    y_pred = np.asarray(y_pred, dtype=int)
    subtypes = np.asarray(subtypes)
    out = {}
    for name in ["FTP-Patator", "SSH-Patator"]:
        mask = subtypes == name
        out[name] = {
            "support": int(mask.sum()),
            "recall": float(y_pred[mask].mean()) if mask.any() else None,
        }
    return out


# Kiểm tra nhanh 2 split chính — chỉ Train/Val
for sc in SCENARIOS:
    Xtr, ytr, _ = load_xy(sc["split"], sc["scenario"], "train")
    Xva, yva, sva = load_xy(sc["split"], sc["scenario"], "validation")
    print(
        sc["job"], sc["split"] + "/" + sc["scenario"] + ":",
        "train", Xtr.shape, "attack=%.4f" % ytr.mean(), "|",
        "val", Xva.shape, "attack=%.4f" % yva.mean(), "|",
        "val subtypes", sva.value_counts().to_dict(),
    )
    print("  Destination Port in X?", "Destination Port" in Xtr.columns)

W3-03 time/with_port: train (90273, 67) attack=0.0569 | val (203826, 67) attack=0.0211 | val subtypes {'BENIGN': 199522, 'FTP-Patator': 2418, 'SSH-Patator': 1886}
  Destination Port in X? True
W3-04 random/with_port: train (91852, 67) attack=0.0310 | val (207465, 67) attack=0.0310 | val subtypes {'BENIGN': 201028, 'FTP-Patator': 3693, 'SSH-Patator': 2744}
  Destination Port in X? True


## Tune RF trên Validation

Mỗi cấu hình: fit Train → predict Validation → chọn theo `f1_attack` (cùng tiêu chí `configs/experiment.yaml`). Không đụng Test.

Chạy khoảng 9 cấu hình × 2 split. Máy desktop thường hết trong 15–40 phút.

In [3]:
def make_rf(n_estimators, max_depth, min_samples_leaf):
    return RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    )


def tune(split, scenario):
    Xtr, ytr, _ = load_xy(split, scenario, "train")
    Xva, yva, sva = load_xy(split, scenario, "validation")
    rows = []
    combos = list(itertools.product(GRID["n_estimators"], GRID["max_depth"], GRID["min_samples_leaf"]))
    for i, (ne, md_, msl) in enumerate(combos, 1):
        t0 = time.time()
        model = make_rf(ne, md_, msl)
        model.fit(Xtr, ytr)
        pred = model.predict(Xva)
        scores = model.predict_proba(Xva)[:, 1]
        m = attack_metrics(yva, pred, scores)
        m.update({
            "n_estimators": ne,
            "max_depth": md_ if md_ is not None else "None",
            "min_samples_leaf": msl,
            "fit_seconds": round(time.time() - t0, 1),
            "split": split,
            "scenario": scenario,
        })
        m["subtype"] = subtype_recall(sva, pred)
        rows.append(m)
        print(f"  [{i}/{len(combos)}] ne={ne} depth={md_} leaf={msl} "
              f"f1={m['f1_attack']:.4f} rec={m['recall_attack']:.4f} "
              f"sec={m['fit_seconds']}")
    df = pd.DataFrame(rows).sort_values(
        ["f1_attack", "average_precision"], ascending=False
    ).reset_index(drop=True)
    best = df.iloc[0]
    # refit best để lưu model
    depth = None if best["max_depth"] == "None" else int(best["max_depth"])
    best_model = make_rf(int(best["n_estimators"]), depth, int(best["min_samples_leaf"]))
    best_model.fit(Xtr, ytr)
    return df, best, best_model


all_grids = {}
all_best = {}
all_models = {}
for sc in SCENARIOS:
    key = f"{sc['split']}_{sc['scenario']}"
    print(f"\n===== {sc['job']}: {key} =====")
    grid, best, model = tune(sc["split"], sc["scenario"])
    all_grids[key] = grid
    all_best[key] = best
    all_models[key] = model
    grid.to_csv(OUT / f"rf_grid_{key}.csv", index=False)
    print("BEST:", {k: best[k] for k in ["n_estimators", "max_depth", "min_samples_leaf", "f1_attack", "recall_attack", "precision_attack"]})


===== W3-03: time_with_port =====
  [1/9] ne=100 depth=10 leaf=5 f1=0.7181 rec=0.5602 sec=2.4
  [2/9] ne=100 depth=16 leaf=5 f1=0.7181 rec=0.5602 sec=2.3
  [3/9] ne=100 depth=None leaf=5 f1=0.7181 rec=0.5602 sec=2.1
  [4/9] ne=150 depth=10 leaf=5 f1=0.7181 rec=0.5602 sec=2.6
  [5/9] ne=150 depth=16 leaf=5 f1=0.7181 rec=0.5602 sec=2.6
  [6/9] ne=150 depth=None leaf=5 f1=0.7181 rec=0.5602 sec=2.8
  [7/9] ne=300 depth=10 leaf=5 f1=0.7181 rec=0.5602 sec=5.1
  [8/9] ne=300 depth=16 leaf=5 f1=0.7181 rec=0.5602 sec=5.2
  [9/9] ne=300 depth=None leaf=5 f1=0.7181 rec=0.5602 sec=5.2
BEST: {'n_estimators': 100, 'max_depth': 16, 'min_samples_leaf': 5, 'f1_attack': 0.7180938198064035, 'recall_attack': 0.5601765799256505, 'precision_attack': 1.0}

===== W3-04: random_with_port =====
  [1/9] ne=100 depth=10 leaf=5 f1=0.9982 rec=0.9994 sec=2.8
  [2/9] ne=100 depth=16 leaf=5 f1=0.9997 rec=0.9994 sec=2.9
  [3/9] ne=100 depth=None leaf=5 f1=0.9997 rec=0.9994 sec=2.9
  [4/9] ne=150 depth=10 leaf=5 f1=0.9

In [4]:
# Bảng Validation — deliverable W3-03 / W3-04
summary_rows = []
for sc in SCENARIOS:
    key = f"{sc['split']}_{sc['scenario']}"
    b = all_best[key]
    row = {
        "job": sc["job"],
        "split": sc["split"],
        "scenario": sc["scenario"],
        "n_estimators": int(b["n_estimators"]),
        "max_depth": b["max_depth"],
        "min_samples_leaf": int(b["min_samples_leaf"]),
        "f1_attack": round(float(b["f1_attack"]), 4),
        "precision_attack": round(float(b["precision_attack"]), 4),
        "recall_attack": round(float(b["recall_attack"]), 4),
        "accuracy": round(float(b["accuracy"]), 4),
        "average_precision": round(float(b["average_precision"]), 4),
        "roc_auc": round(float(b["roc_auc"]), 4),
        "fpr": round(float(b["fpr"]), 4),
        "ftp_recall": b["subtype"]["FTP-Patator"]["recall"],
        "ssh_recall": b["subtype"]["SSH-Patator"]["recall"],
        "fit_seconds": b["fit_seconds"],
    }
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / "rf_validation_results.csv", index=False)

params = {
    "model": "RandomForestClassifier",
    "class_weight": "balanced",
    "max_features": "sqrt",
    "seed": SEED,
    "selection": "validation f1_attack then average_precision",
    "test_opened": False,
    "grid": GRID,
    "timebased": summary_rows[0],
    "random": summary_rows[1],
    "note": (
        "Train time-based has FTP-Patator only; Test has SSH-Patator only. "
        "Primary result is time/with_port. Random split is control."
    ),
}
(OUT / "rf_final_params.json").write_text(json.dumps(params, indent=2, ensure_ascii=False), encoding="utf-8")

# Lưu model đã chọn (joblib) cho TV1 gộp pipeline W3-08 — chưa phải model Test
try:
    import joblib
    for key, model in all_models.items():
        joblib.dump(model, OUT / f"rf_{key}.joblib")
except ImportError:
    print("joblib chưa cài — bỏ qua lưu .joblib")

print("Đã ghi vào", OUT)
summary

Đã ghi vào c:\Users\Dung\cicids-bruteforce-ids\artifacts\TV3_random_forest


,job,split,scenario,n_estimators,max_depth,min_samples_leaf,f1_attack,precision_attack,recall_attack,accuracy,average_precision,roc_auc,fpr,ftp_recall,ssh_recall,fit_seconds
0,W3-03,time,with_port,100,16,5,0.7181,1.0,0.5602,0.9907,0.7326,0.8892,0.0,0.997105,0.000000,2.3
1,W3-04,random,with_port,100,None,5,0.9997,1.0,0.9994,1.0000,0.9998,0.9999,0.0,0.999729,0.998907,2.9


## Test — KHÔNG chạy

Cell dưới cố ý không load Test. TV5 chạy 1 lần ở W3-09 sau khi nhóm chốt `rf_final_params.json`.

In [5]:
print("Chưa tới W3-09 — không đọc X_test / y_test.")
print("File tham số đã chốt:", OUT / "rf_final_params.json")
print("Nộp cho TV1/TV5: rf_validation_results.csv + rf_final_params.json + 2 file rf_grid_*.csv")

Chưa tới W3-09 — không đọc X_test / y_test.
File tham số đã chốt: c:\Users\Dung\cicids-bruteforce-ids\artifacts\TV3_random_forest\rf_final_params.json
Nộp cho TV1/TV5: rf_validation_results.csv + rf_final_params.json + 2 file rf_grid_*.csv
